# Exercise 3: Building a Graph-Based Workflow

In Exercises 1 and 2 you built and chatted with individual host agents. In this exercise you will **wire those agents into a directed graph** — a workflow where each stage automatically feeds the next, and the framework handles routing, parallelism, and output collection for you.

By the end of this exercise you will know how to:

| Concept | What you'll do |
|---|---|
| `Executor` + `@handler` | Define a processing node with a typed message handler |
| `@executor` decorator | Create a lightweight function-based executor |
| `WorkflowBuilder` | Assemble executors into a directed graph |
| `add_edge()` | Define how messages flow between nodes |
| `workflow.run()` | Execute the graph and collect results |
| `AgentExecutor` | Drop an AI agent into the graph as a first-class node |
| Conditional edges | Route messages down different paths at runtime |

<br>

> **Kernel reminder** — make sure you're running the **Workshop (Python 3.12)** kernel. Open the kernel picker (top-right of the notebook in VS Code) and select it under *Jupyter Kernels*. If you don't see it, run **Python: Select Interpreter** from the Command Palette (`Ctrl+Shift+P`) and pick the interpreter at `/opt/venv/bin/python` first.

In [ ]:
from pathlib import Path
from typing import Any, Never
from dataclasses import dataclass

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
)

from utils import load_env, create_agent, AgentOptions, move_to_repo_root

move_to_repo_root()
load_env()  # reads .env at the repo root

---
## 1. Workflows vs. agents

An **agent** is LLM-driven: you hand it a message and it decides — dynamically, turn by turn — which tools to call and what to say next.

```mermaid
flowchart LR
  subgraph Agent["Agent — dynamic, open-ended"]
    msg([message]) --> LLM["LLM / tools"] --> reply([reply])
  end
```

A **workflow** is graph-driven: you explicitly define the nodes (executors) and the connections between them (edges). The framework then executes those nodes in order, passing the output of each one as the input to the next. The graph topology is fixed at build time; what changes at runtime is only the data flowing through it.

```mermaid
flowchart LR
  subgraph Workflow["Workflow — fixed graph, deterministic routing"]
    in([input]) --> A[Executor A]
    A --> B[Executor B]
    A --> C[Executor C]
    B --> out([output])
    C --> out
  end
```

Workflows shine when:
- The steps/stages are **known in advance** 
- You need **parallel execution** 
- You need **conditional branching** 
- You want **type-checked message passing** between stages

---
## 2. The core building block: Executors

An **executor** is a single node in the workflow graph. It receives a typed message, does some work, and either:
- sends a new message downstream with `ctx.send_message(...)`, or
- yields a final result to the caller with `ctx.yield_output(...)`.

Define an executor by decorating an async function with `@executor`:

```python
@executor(id='my-executor')
async def my_executor(message: InputType, ctx: WorkflowContext[OutputType]) -> None:
    await ctx.send_message(...)   # to the next node
    # OR
    await ctx.yield_output(...)  # as a workflow result
```

The decorated function *is* the executor — no class or separate instantiation needed. The `id` is how the framework identifies this node in logs and events.

The `WorkflowContext` type parameter tells the framework what type of message this handler produces:
- `WorkflowContext[str]` — sends a `str` to the next node
- `WorkflowContext[Never, str]` — only yields output to the caller (no downstream message)
- `WorkflowContext` — no output (side-effects only, e.g. logging)

In [ ]:
@executor(id='topic-formatter')
async def formatter(topic: str, ctx: WorkflowContext[str]) -> None:
    """Formats a raw topic string into a structured research prompt."""
    prompt = (
        f"Research question for a Lost podcast episode: {topic}\n"
        "Provide 2-3 paragraphs of relevant background from the show."
    )
    await ctx.send_message(prompt)


@executor(id='topic-printer')
async def topic_printer(text: str, ctx: WorkflowContext[Never, str]) -> None:
    print(text)
    await ctx.yield_output(text)

---
## 3. Wiring executors together: `WorkflowBuilder`

`WorkflowBuilder` assembles executors into a graph. You always:
1. Pass the **starting executor** to the constructor
2. Call `add_edge(source, target)` for each connection
3. Call `build()` to get a runnable `Workflow`

Run the workflow with `await workflow.run(input)`. The input must match the type expected by the start executor's handler.

Collect results with `events.get_outputs()` — it returns a list of everything passed to `ctx.yield_output(...)` anywhere in the graph.

In [ ]:
# Build a 2-node linear workflow: formatter → printer
first_workflow = (
    WorkflowBuilder(start_executor=formatter, output_from='all')
    .add_edge(formatter, topic_printer)
    .build()
)

# Run it — the string goes into the formatter, which sends a prompt to the printer,
# which yields it as output.
events = await first_workflow.run("The hatch — what was really inside it?")

print("\nWorkflow outputs:")
for output in events.get_outputs():
    print(output)

---
## 4. Streaming execution

Pass `stream=True` to `workflow.run()` to receive events as they are emitted rather than waiting for the whole workflow to finish. This is especially useful when the workflow contains AI agents that produce text incrementally.

Each event has a `type` field:
- `"output"` — a final result from `ctx.yield_output(...)`
- `"intermediate"` — a progress event (requires `intermediate_output_from` on the builder)

The streaming and non-streaming modes produce identical *results* — only the timing is different.

In [ ]:
# Streaming version of the same workflow
async for event in first_workflow.run("Why can't anyone leave the island?", stream=True):
    if event.type == "output":
        print(f"[output] {event.data}")

---
## 5. Agents in the graph: `AgentExecutor`

`AgentExecutor` wraps any agent-framework Agent and turns it into a workflow node.

```python
AgentExecutor(agent, id="unique_id")
```

The `id` is how the framework identifies this node in logs and events.

**Input types** — `AgentExecutor` accepts three input types:

| Input | When to use |
|---|---|
| `AgentExecutorRequest` | Starting a fresh turn — you control the prompt |
| `AgentExecutorResponse` | Chaining directly from another agent — conversation history is preserved |
| `str` | Convenience shorthand, but loses prior conversation context |

**Output** — always `AgentExecutorResponse`. Extract the text with:
```python
response.agent_response.text
```

**Bridge executors** — you need one when you want to *reshape* the message before the next node sees it. For example, wrapping a researcher's raw output in a specific prompt before the host reads it:

```python
@executor(id='research-to-host')
async def research_to_host(response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
    await ctx.send_message(AgentExecutorRequest(
        messages=[Message(role="user", contents=[
            f"Here is the research:\n\n{response.agent_response.text}\n\nWhat's your angle?"
        ])],
        should_respond=True,
    ))
```

If the next agent can work with the previous agent's raw output unchanged, you can wire them directly with no bridge.

In [ ]:
# Set your host file names here
HOST_1_FILENAME = "host-joe.md"
HOST_2_FILENAME = "host-maxine.md"


In [ ]:
# Load the host definitions created in Exercise 1
import os

from agent_framework import Agent
from agent_framework_foundry import FoundryChatClient
from azure.core.credentials_async import AsyncTokenCredential
from azure.core.credentials import AccessToken


api_key = os.getenv("FOUNDRY_API_KEY")

class _ApiKeyCredential(AsyncTokenCredential):
    """Wraps an API key as a TokenCredential for Foundry."""
    async def get_token(self, *scopes, **kwargs):
        return AccessToken(api_key, 0) # type: ignore
    async def close(self): pass
    async def __aenter__(self): return self
    async def __aexit__(self, *args): pass

credential = _ApiKeyCredential()

host_1_instructions = Path(f"output/agents/{HOST_1_FILENAME}").read_text()
host_2_instructions  = Path(f"output/agents/{HOST_2_FILENAME}").read_text()

client = FoundryChatClient(
    project_endpoint=os.getenv("FOUNDRY_PROJECT_ENDPOINT"),
    model=os.getenv("FOUNDRY_MODEL"),
    credential=credential,
)

host_1_agent = Agent(
    client=client,  # type: ignore
    instructions=host_1_instructions,
)

host_2_agent = Agent(
    client=client,  # type: ignore
    instructions=host_2_instructions,
)

# Wrap each host agent in an AgentExecutor — they become nodes in the graph
host_1_executor = AgentExecutor(
    host_1_agent,
    id="host 1",
)

host_2_executor = AgentExecutor(
    host_2_agent,
    id="host 2",
)


@executor(id='response-printer')
async def response_printer(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(response.agent_response.text)

In [ ]:
# Minimal single-agent workflow: host_1 → response_printer
single_agent_workflow = (
    WorkflowBuilder(start_executor=host_1_executor, output_from=[response_printer])
    .add_edge(host_1_executor, response_printer)
    .build()
)

# The AgentExecutor will receive an AgentExecutorRequest
request = AgentExecutorRequest(
    messages=[Message(role="user", contents=["What's your honest take on the polar bear in Lost?"])],
    should_respond=True,
)

events = await single_agent_workflow.run(request)
for output in events.get_outputs():
    print(f"Workflow Output: {output}")

---
## 6. A podcast research pipeline

Now let's build a proper multi-stage pipeline. A simple pre-production workflow for a podcast episode looks like this:

```mermaid
flowchart TD
  topic([topic str]) --> re["research_executor\nAI agent: researches the topic"]
  re --> rth["research_to_host\nbridge: AgentExecutorResponse<br>→<br> AgentExecutorRequest"]
  rth --> h1["host_1_executor\nAI agent: forms their angle on the research"]
  h1 -->|final output| out([AgentResponse])
```

The **bridge executor** `research_to_host` is the glue between the two agents. It converts the researcher's raw `AgentExecutorResponse` into a shaped `AgentExecutorRequest` for the host, and also yields the message text as an **intermediate output** so callers can inspect what was sent.

In [ ]:
# A generic research agent
research_executor = AgentExecutor(
    create_agent(AgentOptions(
        name="Researcher",
        instructions=(
            f"You are a research assistant for a podcast about a topic. "
            "Given a topic, return 2-3 paragraphs of factual background for the show. "
            "Be concise and accurate."
        ),
    )),
    id="researcher",
)

# Bridge: researcher response → AgentExecutorRequest that gives host context
# takes in ctx: WorkflowContext[AgentExecutorRequest, str] because it is outputting
# 1. an AgentExecutorRequest, sent to the downstream node - host_1_executor (an agent)
# 2. a String (str) of the message sent to the host agent executor as intermediate output
@executor(id='research-to-host')
async def research_to_host(response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorRequest, str]) -> None:
    research_text = response.agent_response.text
    message_to_host = (
        f"Here is research on today's topic:\n\n{research_text}\n\n"
        "Based on this, what's your angle for the episode?"
    )
    await ctx.yield_output(message_to_host)   # ← emits the message as intermediate output
    await ctx.send_message(
        AgentExecutorRequest(
            messages=[Message(role="user", contents=[message_to_host])],
            should_respond=True,
        )
    )


# 2-node research pipeline
research_pipeline = (
    WorkflowBuilder(start_executor=research_executor, output_from=[host_1_executor], intermediate_output_from=[research_to_host])
    .add_edge(research_executor, research_to_host)
    .add_edge(research_to_host, host_1_executor)
    .build()
)

In [ ]:
events = await research_pipeline.run('The TV Show LOST')
for msg in events.get_intermediate_outputs():
    print(f"\n[Message to Host]\n{msg}")
for resp in events.get_outputs():
    print(f"\n[Host Response]\n{resp.text}")


---
## 7. Conditional edges

Sometimes the right path through the graph depends on the data. You can attach a **condition function** to an edge — the framework only traverses that edge if the function returns `True` for the message the upstream executor sent.

```python
builder.add_edge(source, target, condition=my_condition_fn)
```

In the podcast context, host 1 owns plot/mystery topics and host 2 owns character psychology. The workflow below classifies the topic first, then routes to the right host.

```mermaid
flowchart TD
  topic([topic str]) --> tc[topic_classifier]
  tc -->|PlotTopic| pth[plot_to_host1]
  tc -->|CharacterTopic| cth[char_to_host2]
  pth --> h1[host_1_executor]
  cth --> h2[host_2_executor]
  h1 -->|final output| out1([AgentResponse])
  h2 -->|final output| out2([AgentResponse])
```

By using **typed dataclasses** as the message output of `topic_classifier`, the condition functions become simple `isinstance` checks — readable and easy to extend.

In [ ]:
@dataclass
class PlotTopic:
    topic: str

@dataclass
class CharacterTopic:
    topic: str


_CHARACTER_KEYWORDS = [
    "jack", "locke", "sawyer", "kate", "hurley",
    "desmond", "ben", "richard", "character", "psychology",
]


@executor(id='topic-classifier')
async def topic_classifier(topic: str, ctx: WorkflowContext[PlotTopic|CharacterTopic,str]) -> None:
    if any(kw in topic.lower() for kw in _CHARACTER_KEYWORDS):
        await ctx.yield_output("→ CHARACTER topic — routing to host 2")   # ← emits the topic as intermediate output
        await ctx.send_message(CharacterTopic(topic=topic))
    else:
        await ctx.yield_output("→ PLOT/MYSTERY topic — routing to host 1")   # ← emits the topic as intermediate output
        await ctx.send_message(PlotTopic(topic=topic))


# Bridge executors — each converts the typed topic into an AgentExecutorRequest
@executor(id='plot-to-host1')
async def plot_to_host1(pt: PlotTopic, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
    await ctx.send_message(AgentExecutorRequest(
        messages=[Message(role="user", contents=[f"Your plot/mystery angle on: {pt.topic}"])],
        should_respond=True,
    ))


@executor(id='char-to-host2')
async def char_to_host2(ct: CharacterTopic, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
    await ctx.send_message(AgentExecutorRequest(
        messages=[Message(role="user", contents=[f"Your character psychology angle on: {ct.topic}"])],
        should_respond=True,
    ))


# Condition functions — each receives the message the classifier sent
def is_plot_topic(msg: Any) -> bool:
    return isinstance(msg, PlotTopic)

def is_character_topic(msg: Any) -> bool:
    return isinstance(msg, CharacterTopic)


# Build the conditional workflow
conditional_workflow = (
    WorkflowBuilder(start_executor=topic_classifier, output_from=[host_1_executor, host_2_executor], intermediate_output_from=[topic_classifier])
    .add_edge(topic_classifier, plot_to_host1,  condition=is_plot_topic)
    .add_edge(topic_classifier, char_to_host2,  condition=is_character_topic)
    .add_edge(plot_to_host1, host_1_executor)
    .add_edge(char_to_host2, host_2_executor)
    .build()
)

In [ ]:
# A character-focused topic — should route to host 2
events = await conditional_workflow.run(
    "Locke's relationship with his father — how does it shape his arc on the island?"
)
for event in events:
    if event.type == "intermediate":
        print(f"\n[Intermediate Output]: {event.data}\n")
    elif event.type == "output":
        print(f"\n[Output]: {event.data}\n")


In [ ]:
# A plot/mystery topic — should route to  — should route to host 1
events = await conditional_workflow.run(
    "Why does the island need to be protected, and from what exactly?"
)
for event in events:
    if event.type == "intermediate":
        print(f"\n[Intermediate Output]: {event.data}\n")
    elif event.type == "output":
        print(f"\n[Output]: {event.data}\n")

---
## 8. Challenge: fan-out to both hosts in parallel

So far, each topic has gone to only one host. But a real podcast needs *both* perspectives. The graph API supports **fan-out**: if you add multiple edges from the same source executor, the framework sends the message to *all* connected targets and runs them in the same superstep — in parallel.

```mermaid
flowchart TD
  topic([topic str]) --> re[research_executor]
  re --> rth[research_to_host]
  rth --> h1[host_1_executor]
  rth --> h2[host_2_executor]
  h1 --> eb[episode_brief]
  h2 --> eb[episode_brief]
  h1~~~h2:::parallel
  classDef parallel fill:none,stroke:none
```

Both hosts run simultaneously. Each one yields its output independently, so `events.get_outputs()` returns two items — one per host. Try changing `EPISODE_TOPIC` to something divisive and see how each host frames it differently.

In [ ]:
@executor(id='episode-brief')
async def episode_brief(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(response.agent_response.text)


# Fan-out workflow: research_to_host sends the same request to BOTH hosts simultaneously
fanout_workflow = (
    WorkflowBuilder(start_executor=research_executor, output_from=[episode_brief])
    .add_edge(research_executor, research_to_host)
    # Fan-out: two edges from the same source — both hosts run in parallel
    .add_edge(research_to_host, host_1_executor)
    .add_edge(research_to_host, host_2_executor)
    .add_edge(host_1_executor, episode_brief)
    .add_edge(host_2_executor, episode_brief)
    .build()
)


EPISODE_TOPIC = "The numbers — 4 8 15 16 23 42. What do they really mean?"

print(f"Topic: {EPISODE_TOPIC}")
print("=" * 60)

events = await fanout_workflow.run(EPISODE_TOPIC)

for i, output in enumerate(events.get_outputs(), 1):
    print(f"\n--- Host {i} ---")
    print(output)